# Step 1 — Data Collection & Layer 2 Merge
**PFE: A Machine Learning Approach to Financial News Sentiment and Market Trend Prediction**

**Student:** Khelil Dhiaeddine | **Supervisor:** Nesrine Lahiani

---

## Objective
Normalize all source-specific Layer 1 datasets into a unified Layer 2 Parquet file ready for NLP preprocessing.

### Sources
- `reddit_s1` — HuggingFace Reddit dataset (~1,596 Tesla posts, 2020–2023)
- `reddit_s2_2022` — Reddit dataset with different schema (2022 data)
- `news` — Financial news via NewsAPI/Finnhub (~73,367 articles)
- `twitter` — Pre-collected tweet dataset

### Layer 2 Target Schema
| Field | Type | Description |
|-------|------|-------------|
| `doc_id` | string | Unique document ID |
| `published_at` | datetime UTC | Normalized timestamp |
| `source` | string | Source tag |
| `text` | string | Full text content |
| `ticker` | string | Stock ticker |
| `url` | string | Source URL (nullable) |

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import os
import warnings
warnings.filterwarnings('ignore')

# ── Kaggle paths ──────────────────────────────────────────────────────────────
# If running on Kaggle, your uploaded datasets appear under /kaggle/input/
# Adjust these paths to match your dataset slugs on Kaggle

BASE_INPUT  = '/kaggle/input/pfe-tsla-layer1'   # <-- change to your dataset slug
BASE_OUTPUT = '/kaggle/working/'

# Layer 1 file paths
PATH_REDDIT_S1       = os.path.join(BASE_INPUT, 'reddit_s1.parquet')
PATH_REDDIT_S2       = os.path.join(BASE_INPUT, 'reddit_s2_2022.parquet')
PATH_NEWS            = os.path.join(BASE_INPUT, 'news_raw.parquet')
PATH_TWITTER         = os.path.join(BASE_INPUT, 'twitter_raw.parquet')

# Layer 2 output
PATH_LAYER2_OUTPUT   = os.path.join(BASE_OUTPUT, 'layer2_unified_final.parquet')

TICKER = 'TSLA'

print('Configuration loaded.')
print(f'Output will be saved to: {PATH_LAYER2_OUTPUT}')

## 1. Helper Functions

In [ ]:
def generate_doc_id(row, fields=('published_at', 'text')):
    """
    Generate a deterministic 16-character MD5 doc_id
    from a combination of timestamp + text prefix.
    """
    raw = '_'.join(str(row.get(f, ''))[:50] for f in fields)
    return hashlib.md5(raw.encode()).hexdigest()[:16]


def extract_subreddit_from_url(url):
    """
    Extract subreddit name from a Reddit URL.
    e.g. https://www.reddit.com/r/teslamotors/... -> teslamotors
    """
    try:
        parts = str(url).split('/')
        r_idx = parts.index('r')
        return parts[r_idx + 1]
    except (ValueError, IndexError):
        return None


def to_layer2(df, doc_id_col=None, published_at_col='published_at',
              source_val=None, text_col='text',
              ticker_val=TICKER, url_col=None):
    """
    Normalize any source dataframe into the Layer 2 unified schema:
    [doc_id, published_at, source, text, ticker, url]
    """
    out = pd.DataFrame()

    # doc_id
    if doc_id_col and doc_id_col in df.columns:
        out['doc_id'] = df[doc_id_col].astype(str)
    else:
        out['doc_id'] = df.apply(generate_doc_id, axis=1)

    # published_at
    out['published_at'] = df[published_at_col]

    # source
    out['source'] = source_val if source_val else df['source']

    # text
    out['text'] = df[text_col]

    # ticker
    out['ticker'] = ticker_val

    # url
    out['url'] = df[url_col] if (url_col and url_col in df.columns) else None

    return out


def schema_report(df, name):
    print(f'\n── {name} ──────────────────────────────')
    print(df.dtypes)
    print(df.notna().sum())
    print(f'Total rows: {len(df)}')


print('Helper functions defined.')

## 2. Reddit S1 — HuggingFace Dataset

In [ ]:
df_r1 = pd.read_parquet(PATH_REDDIT_S1)
print('Reddit S1 raw shape:', df_r1.shape)
print(df_r1.columns.tolist())
df_r1.head(3)

In [ ]:
# Reddit S1 — Layer 1 schema already standard
# Fields: id, title, selftext, score, upvote_ratio, num_comments,
#         subreddit, created_utc, url, ticker, source

# Build combined text: title + selftext
def build_reddit_text(row):
    title    = str(row.get('title', '')).strip()    if pd.notna(row.get('title'))    else ''
    selftext = str(row.get('selftext', '')).strip() if pd.notna(row.get('selftext')) else ''
    if title and selftext:
        return title + '. ' + selftext
    return title or selftext

df_r1['combined_text'] = df_r1.apply(build_reddit_text, axis=1)

# Drop rows with no text
df_r1 = df_r1[df_r1['combined_text'].str.strip() != '']

# Normalize to Layer 2
df_r1_l2 = to_layer2(
    df_r1,
    doc_id_col      = 'id',
    published_at_col= 'created_utc',
    source_val      = 'reddit',
    text_col        = 'combined_text',
    url_col         = 'url'
)

schema_report(df_r1_l2, 'Reddit S1 → Layer 2')

## 3. Reddit S2 — 2022 Dataset (Different Schema)

In [ ]:
df_r2 = pd.read_parquet(PATH_REDDIT_S2)
print('Reddit S2 raw shape:', df_r2.shape)
print(df_r2.columns.tolist())
df_r2.head(3)

In [ ]:
# Reddit S2 schema differences:
# - 'body'       -> maps to selftext
# - 'comms_num'  -> maps to num_comments
# - 'created'    -> two timestamp cols with ~7200s discrepancy (UTC vs EST)
# - 'subreddit'  -> missing, extract from URL
# - 'author'     -> missing

# Rename to standard names
df_r2 = df_r2.rename(columns={
    'body'      : 'selftext',
    'comms_num' : 'num_comments',
})

# Resolve timestamp: use 'created_utc' if present, else 'created' (treat as UTC)
if 'created_utc' in df_r2.columns:
    df_r2['timestamp'] = df_r2['created_utc']
elif 'created' in df_r2.columns:
    # 'created' is likely UNIX epoch in local time — convert to UTC
    df_r2['timestamp'] = pd.to_datetime(df_r2['created'], unit='s', utc=True)

# Extract subreddit from URL if missing
if 'subreddit' not in df_r2.columns:
    df_r2['subreddit'] = df_r2['url'].apply(extract_subreddit_from_url)

# Build combined text
df_r2['combined_text'] = df_r2.apply(build_reddit_text, axis=1)
df_r2 = df_r2[df_r2['combined_text'].str.strip() != '']

# Normalize to Layer 2
df_r2_l2 = to_layer2(
    df_r2,
    published_at_col = 'timestamp',
    source_val       = 'reddit',
    text_col         = 'combined_text',
    url_col          = 'url'
)

schema_report(df_r2_l2, 'Reddit S2 → Layer 2')

## 4. News Dataset

In [ ]:
df_news = pd.read_parquet(PATH_NEWS)
print('News raw shape:', df_news.shape)
print(df_news.columns.tolist())
print(df_news[['headline', 'content']].notna().sum())
df_news.head(3)

In [ ]:
# News Layer 1 schema:
# article_id, published_at, source_name, author, headline, content, url, ticker, source
#
# Issue: 29,687 rows where both headline and content are null
# (API collected metadata only, never fetched article text)
# Decision: drop those rows, concatenate headline + content for the rest

# Drop rows where both headline and content are null
df_news = df_news[df_news['headline'].notna() | df_news['content'].notna()]
print(f'After dropping null-text rows: {len(df_news)} rows')

# Concatenate headline + content (Option C)
def build_news_text(row):
    headline = str(row['headline']).strip() if pd.notna(row['headline']) else ''
    content  = str(row['content']).strip()  if pd.notna(row['content'])  else ''
    if headline and content:
        return headline + '. ' + content
    return headline or content

df_news['text'] = df_news.apply(build_news_text, axis=1)

# Normalize to Layer 2
df_news_l2 = to_layer2(
    df_news,
    doc_id_col       = 'article_id',
    published_at_col = 'published_at',
    source_val       = 'news',
    text_col         = 'text',
    url_col          = 'url'
)

schema_report(df_news_l2, 'News → Layer 2')

## 5. Twitter Dataset

In [ ]:
df_tw = pd.read_parquet(PATH_TWITTER)
print('Twitter raw shape:', df_tw.shape)
print(df_tw.columns.tolist())
df_tw.head(3)

In [ ]:
# Twitter Layer 1 schema:
# tweet_id, created_at, text, username, retweet_count, like_count, ticker, source
#
# Issue: doc_id was null for all 37,422 twitter rows in Layer 2
# Fix: generate deterministic doc_id from tweet_id if available, else hash

if 'tweet_id' in df_tw.columns:
    df_tw['doc_id'] = df_tw['tweet_id'].astype(str)
else:
    df_tw['doc_id'] = df_tw.apply(
        lambda row: hashlib.md5(
            f"{row.get('created_at', '')}_{str(row.get('text', ''))[:50]}".encode()
        ).hexdigest()[:16],
        axis=1
    )

# Detect timestamp column
ts_col = 'created_at' if 'created_at' in df_tw.columns else 'published_at'

# Drop rows with no text
df_tw = df_tw[df_tw['text'].notna() & (df_tw['text'].str.strip() != '')]

# Normalize to Layer 2
df_tw_l2 = to_layer2(
    df_tw,
    doc_id_col       = 'doc_id',
    published_at_col = ts_col,
    source_val       = 'twitter',
    text_col         = 'text',
    url_col          = None
)

schema_report(df_tw_l2, 'Twitter → Layer 2')

## 6. Concatenate All Sources → Layer 2

In [ ]:
frames = [df_r1_l2, df_r2_l2, df_news_l2, df_tw_l2]

df_layer2 = pd.concat(frames, ignore_index=True)

print('Combined shape before cleaning:', df_layer2.shape)
print(df_layer2.notna().sum())

## 7. Final Cleaning

In [ ]:
# ── 7.1 Normalize published_at to UTC datetime ────────────────────────────────
# Mixed types: some sources already have Timestamps, others have raw strings
df_layer2['published_at'] = pd.to_datetime(
    df_layer2['published_at'], utc=True, errors='coerce'
)

print('published_at dtype:', df_layer2['published_at'].dtype)
print('published_at nulls after parse:', df_layer2['published_at'].isna().sum())

In [ ]:
# ── 7.2 Drop rows with no text (safety net) ───────────────────────────────────
before = len(df_layer2)
df_layer2 = df_layer2.dropna(subset=['text'])
df_layer2 = df_layer2[df_layer2['text'].str.strip() != '']
print(f'Dropped {before - len(df_layer2)} null/empty text rows')

# ── 7.3 Deduplicate on doc_id ─────────────────────────────────────────────────
before = len(df_layer2)
df_layer2 = df_layer2.drop_duplicates(subset=['doc_id'])
print(f'Dropped {before - len(df_layer2)} duplicate doc_id rows')

# ── 7.4 Filter to target date range ───────────────────────────────────────────
df_layer2 = df_layer2[
    (df_layer2['published_at'] >= '2020-01-01') &
    (df_layer2['published_at'] <= '2023-12-31')
]
print(f'After date filter: {len(df_layer2)} rows')

In [ ]:
# ── 7.5 Final schema report ───────────────────────────────────────────────────
print('\n══ FINAL LAYER 2 REPORT ══')
print(df_layer2.dtypes)
print()
print(df_layer2.notna().sum())
print(f'\nTotal rows : {len(df_layer2):,}')
print('\nSource distribution:')
print(df_layer2['source'].value_counts())
print('\nTicker distribution:')
print(df_layer2['ticker'].value_counts())
print('\nDate range:')
print('From:', df_layer2['published_at'].min())
print('To  :', df_layer2['published_at'].max())

## 8. Save Layer 2 Final Parquet

In [ ]:
df_layer2.to_parquet(PATH_LAYER2_OUTPUT, index=False)
print(f'Saved: {PATH_LAYER2_OUTPUT}')
print(f'File size: {os.path.getsize(PATH_LAYER2_OUTPUT) / 1e6:.2f} MB')

# Verify round-trip
df_verify = pd.read_parquet(PATH_LAYER2_OUTPUT)
print(f'Verification read: {len(df_verify):,} rows — OK' if len(df_verify) == len(df_layer2) else 'WARNING: row count mismatch!')

## 9. Daily Document Count — Quick EDA
Checking density across trading days before moving to NLP.

In [ ]:
import matplotlib.pyplot as plt

df_layer2['date'] = df_layer2['published_at'].dt.date

daily_counts = df_layer2.groupby(['date', 'source']).size().unstack(fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Total daily volume
daily_counts.sum(axis=1).rolling(7).mean().plot(ax=axes[0], color='steelblue')
axes[0].set_title('Daily Document Count (7-day rolling average)')
axes[0].set_ylabel('Documents')
axes[0].set_xlabel('')

# By source
daily_counts.rolling(30).mean().plot(ax=axes[1])
axes[1].set_title('Daily Document Count by Source (30-day rolling average)')
axes[1].set_ylabel('Documents')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.savefig('/kaggle/working/daily_document_counts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

---
## ✅ Step 1 Complete

| Output | Value |
|--------|-------|
| Final rows | ~87,196 |
| Null text | 0 |
| Null doc_id | 0 |
| Date range | 2020-01-01 → 2023-12-31 |
| Output file | `layer2_unified_final.parquet` |

**Next step:** `02_eda.ipynb` — Exploratory Data Analysis before NLP Preprocessing.